In [7]:
# Import necessary libraries
import os
import sys
import shutil
import glob

import numpy as np
import yaml
from tqdm.contrib.concurrent import thread_map# or thread_map

In [8]:
def create_project_file(file_pth: str, dataset_pth: str, label_mapping: dict) -> None:
    data = {
        "path": dataset_pth,
        "train": "images/train",
        "val": "images/val",
        "kpt_shape": [1, 2],
    }

    with open(file_pth, "wt") as f:
        yaml.dump(data, f, sort_keys=False, default_flow_style=None)
    
    with open(file_pth, "at") as f:
        yaml.dump({"names": label_mapping}, f, sort_keys=False)

In [9]:
def get_files_glob(directory, ext="*.txt"):
    pattern = os.path.join(directory, '**', ext)
    txt_files = glob.glob(pattern, recursive=True)
    return txt_files

In [10]:
MARGIN = 0.15

BASE_PTH = "/home/kyamkovyi/projects/researches/people_detection_pg2/quality_reduce_comparison"
BASE_DATASET = "cam4_poc_part1"
FILTERED_DATASET = f"cam4_poc_part1_{int(MARGIN * 100)}"

SUBSETS = ["train", "val"]

In [11]:
src_pth = os.path.join(BASE_PTH, BASE_DATASET)
dst_pth = os.path.join(BASE_PTH, FILTERED_DATASET)
if os.path.exists(dst_pth):
    shutil.rmtree(dst_pth)
shutil.copytree(src_pth, dst_pth)

cache_files = get_files_glob(dst_pth, "*.cache")
for i in cache_files:
    os.remove(i)

In [13]:
# create project file
create_project_file(os.path.join(dst_pth, "project.yaml"), dst_pth, {0: "person"})

In [14]:
labels_pths = get_files_glob(os.path.join(dst_pth, "labels"))
len(labels_pths)

152

In [15]:
def get_min_margin(x, y, widths, heights):
    # Calculate the margin sizes
    left_margin = x + widths / 2  # Distance from right side of the box to the left edge of the frame
    right_margin = 1 - (x - widths / 2)  # Distance from left side of the box to the right edge of the frame
    top_margin = y + heights / 2  # Distance from bottom side of the box to the top edge of the frame
    bottom_margin = 1 - (y - heights / 2)  # Distance from top side of the box to the bottom edge of the frame
    
    return np.min([left_margin, right_margin, top_margin, bottom_margin])

def process_file(pth):
    filtered_lines = []

    count = 0
    count_filtered = 0
    with open(pth, "rt") as f:
        for line in f.readlines():
            count +=1
            _, x_c, y_c, w, h, _, _ = [float(i) for i in line.split()]
            
            min_margin = get_min_margin(x_c, y_c, w, h)
            if min_margin > MARGIN:
                filtered_lines.append(line)
                count_filtered += 1

    with open(pth, "wt") as f:
        f.writelines(filtered_lines)

    return count, count_filtered

In [16]:
r = thread_map(process_file, labels_pths, max_workers=8)

count = 0
count_filtered = 0
for i, j in r:
    count += i
    count_filtered += j

print(f"Margin {MARGIN} removes {(count - count_filtered) / count * 100:.02f}% bboxes")

100%|██████████| 152/152 [00:00<00:00, 6732.86it/s]

Margin 0.15 removes 9.06% bboxes
